In [29]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

from evidently import Report
from evidently.metrics import *
from evidently import DataDefinition, Regression, Dataset


In [31]:
df = pd.read_csv("DelayedFlights.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1936758 entries, 0 to 1936757
Data columns (total 30 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Unnamed: 0         int64  
 1   Year               int64  
 2   Month              int64  
 3   DayofMonth         int64  
 4   DayOfWeek          int64  
 5   DepTime            float64
 6   CRSDepTime         int64  
 7   ArrTime            float64
 8   CRSArrTime         int64  
 9   UniqueCarrier      object 
 10  FlightNum          int64  
 11  TailNum            object 
 12  ActualElapsedTime  float64
 13  CRSElapsedTime     float64
 14  AirTime            float64
 15  ArrDelay           float64
 16  DepDelay           float64
 17  Origin             object 
 18  Dest               object 
 19  Distance           int64  
 20  TaxiIn             float64
 21  TaxiOut            float64
 22  Cancelled          int64  
 23  CancellationCode   object 
 24  Diverted           int64  
 25  CarrierDelay      

In [32]:
df = df.dropna(subset=[
    'DepDelay', 'Distance', 'CarrierDelay', 'WeatherDelay',
    'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'ArrDelay'
])

In [34]:
df.rename(columns={"ArrDelay": "target"}, inplace=True)
df['prediction'] = df['target'].values + np.random.normal(0, 10, df.shape[0])
df.head()

,Unnamed: 0,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,...,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,prediction
3,4,2008,1,3,4,1829.0,1755,1959.0,1925,WN,...,10.0,0,N,0,2.0,0.0,0.0,0.0,32.0,31.435964
5,6,2008,1,3,4,1937.0,1830,2037.0,1940,WN,...,7.0,0,N,0,10.0,0.0,0.0,0.0,47.0,47.207969
7,11,2008,1,3,4,1644.0,1510,1845.0,1725,WN,...,8.0,0,N,0,8.0,0.0,0.0,0.0,72.0,61.468558
9,16,2008,1,3,4,1452.0,1425,1640.0,1625,WN,...,8.0,0,N,0,3.0,0.0,0.0,0.0,12.0,9.821752
11,18,2008,1,3,4,1323.0,1255,1526.0,1510,WN,...,9.0,0,N,0,0.0,0.0,0.0,0.0,16.0,32.576740


In [35]:
df_ref = df.sample(n=5000, replace=False)
df_cur = df.sample(n=5000, replace=False)

In [36]:
from evidently import DataDefinition, Regression

data_definition=DataDefinition(
        regression=[Regression(target="target", prediction="prediction")]
    )

In [37]:
from evidently import Dataset

reference_dataset = Dataset.from_pandas(
    pd.DataFrame(df_ref),
    data_definition=data_definition,

)

In [38]:
current_dataset = Dataset.from_pandas(
    pd.DataFrame(df_cur),
    data_definition=data_definition,
)

In [44]:
from evidently import Report
from evidently.metrics import *

report = Report([
    MeanError(),
    MAE(),
    MAPE(),
    RMSE(),
    R2Score(),
    AbsMaxError(),
    DummyMAE(),
    DummyMAPE(),
    DummyRMSE(),
])
report.run(current_dataset, reference_dataset)
report

d:\MLops_ModelQualityReport_1049\venv\Lib\site-packages\sklearn\metrics\_regression.py:1266: UndefinedMetricWarning:

R^2 score is not well-defined with less than two samples.

d:\MLops_ModelQualityReport_1049\venv\Lib\site-packages\sklearn\metrics\_regression.py:1266: UndefinedMetricWarning:

R^2 score is not well-defined with less than two samples.

d:\MLops_ModelQualityReport_1049\venv\Lib\site-packages\sklearn\metrics\_regression.py:1266: UndefinedMetricWarning:

R^2 score is not well-defined with less than two samples.

d:\MLops_ModelQualityReport_1049\venv\Lib\site-packages\sklearn\metrics\_regression.py:1266: UndefinedMetricWarning:

R^2 score is not well-defined with less than two samples.

